# Planning a before/after study: paired t test sample size

**Sample Size and Power Calculation** · a worked example from the StatNex teaching guide

**The story.** Before running a study you must decide how many people to include. A company plans a before/after study of its sales reps. The calculation uses no data, only its settings; Cohen's reference table of "small", "medium" and "large" effects helps to choose the effect size.

**The question.** How many sales reps does a before/after study need so that a real effect is very unlikely to be missed?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import matplotlib
from IPython import get_ipython
from IPython.display import Image, display

matplotlib.use("Agg")
shown = set()


def show_new_figures(*_):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path, path.stat().st_mtime_ns) not in shown:
            shown.add((path, path.stat().st_mtime_ns))
            display(Image(filename=str(path)))


events = get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `13_sample_size/sample_size__effect_sizes.csv` (8 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Test,Measure,Small,Medium,Large,Notes
Difference test (t test),r,0.1,0.3,0.5,The same values apply to the paired t test and the two-sample t test. Two measures are available: r and d.
,d,0.2,0.5,0.8,
Analysis of variance,eta squared,0.01,0.06,0.14,Multiple-comparison methods use r from the difference test (t test)
Correlation,r,0.1,0.3,0.5,The correlation coefficient itself
Chi-square test (2x2 table),phi,0.1,0.3,0.5,The association coefficient
Chi-square test (other tables),Cramer's V,0.1,0.3,0.5,The association coefficient
Difference test (nonparametric test),r,0.1,0.3,0.5,"For the Mann-Whitney, Wilcoxon, Kruskal-Wallis and Friedman tests, get Z from Z = r * sqrt(n)"
Multiple regression,R squared,0.02,0.13,0.26,The coefficient of determination

## Libraries

In [ ]:
import math

import numpy as np
import pandas as pd
from scipy import optimize, stats
from statsmodels.stats.power import TTestIndPower, TTestPower

pd.set_option("display.width", 120)

# No data file: everything comes from the numbers set below.
alpha = 5 / 100
sides = ["two-sided", "one-sided"]


def z(p):
    """The standard normal value with probability p above it."""
    return stats.norm.isf(p)

## Required sample size

In [ ]:
# Per group, as a continuous number; round it up yourself. The formulas follow
# Nagata, "How to decide the sample size" (Asakura), as the app does. For the t tests
# that is a closed-form approximation; statsmodels solves the noncentral t exactly,
# which is printed below as a cross-check.
power = 95 / 100
z_beta = z(1 - power)
effect = 0.5
size = abs(effect)
# One mean (or the mean of paired differences), effect size d = difference / SD.
# The z^2/2 term corrects for estimating the SD.


def required(two_sided):
    z_alpha = z(alpha / 2 if two_sided else alpha)
    return (z_alpha + z_beta) ** 2 / size**2 + z_alpha**2 / 2


def power_at(n, two_sided):
    # Power of the t test at n, from the noncentral t (for the chart).
    df, ncp = n - 1, size * math.sqrt(n)
    if two_sided:
        critical = stats.t.isf(alpha / 2, df)
        return stats.nct.sf(critical, df, ncp) + stats.nct.cdf(-critical, df, ncp)
    return stats.nct.sf(stats.t.isf(alpha, df), df, ncp)


two, one = required(True), required(False)
sample_size = pd.DataFrame([[two, one], [math.ceil(two), math.ceil(one)]],
                           index=["n per group", "rounded up"], columns=sides)
print(sample_size)

# Cross-check: statsmodels solves the t test's power equation exactly. Its n is a little
# larger than the closed form's; rounded up, the two nearly always agree.
solver = TTestPower()
exact = [solver.solve_power(effect_size=size, alpha=alpha, power=power, alternative=side)
         for side in ("two-sided", "larger")]
print(f"\nstatsmodels, exact noncentral t: {exact[0]:.4f} (two-sided), {exact[1]:.4f} (one-sided)")

## Power curve

In [ ]:
# Power against n per group; where a curve crosses the target is the n above.
import os

import matplotlib.pyplot as plt

os.makedirs("figures", exist_ok=True)
grid = np.linspace(2.5, max(two, one) * 2.5, 120)
figure, axis = plt.subplots(figsize=(6, 4))
for side, two_sided in zip(sides, (True, False)):
    axis.plot(grid, [power_at(n, two_sided) for n in grid], label=side)
axis.axhline(power, color="grey", linestyle="--", label="target power")
axis.set(xlabel="sample size per group", ylabel="power")
axis.legend()
figure.savefig("figures/power_curve.png", dpi=120, bbox_inches="tight")
plt.close(figure)

## Benchmarks for effect sizes (Cohen)

In [ ]:
# Rules of thumb for when nothing better is known: plan from the smallest effect that
# matters to the decision instead. The Cramer's V row repeats Cohen's values for w,
# which hold for V only when the smaller side of the table has two categories; for a
# larger table divide them by sqrt(min(rows, columns) - 1).
benchmarks = pd.DataFrame(
    [["t test", "r", 0.1, 0.3, 0.5], ["t test", "d", 0.2, 0.5, 0.8],
     ["ANOVA", "eta^2", 0.01, 0.06, 0.14], ["correlation", "r", 0.1, 0.3, 0.5],
     ["chi-square (2x2)", "phi", 0.1, 0.3, 0.5], ["chi-square (other)", "Cramer's V", 0.1, 0.3, 0.5],
     ["non-parametric", "r", 0.1, 0.3, 0.5], ["multiple regression", "R^2", 0.02, 0.13, 0.26]],
    columns=["analysis", "index", "small", "medium", "large"],
)
print()
print(benchmarks.to_string(index=False))

## What the result says

The calculation needs three settings. The effect size d is the change you want to detect, divided by how much changes vary from person to person: here 0.5. The significance level is the risk you accept of finding an effect that is not there: here 5%. The power is the chance of detecting the effect if it is real: here 95%.

The sample size required is 53.9, rounded up to 54 people for a two-sided test. A one-sided test, which looks for a change in one direction only, needs 45.

The effect size guides label d = 0.5 as medium, 0.2 as small and 0.8 as large. So the study needs 54 reps to be 95% sure of catching a medium effect. That answer is only as good as the effect you plan for.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.